In [1]:
#Code block to connect to API and get the last 2 days of data 
#API infos : 
#import requests
#import os
#api_key = os.environ.get("API_KEY")
#headers = {
#    "Authorization": f"Apikey {api_key}",
#    "Content-Type": "application/json"
#}
#response = requests.get("https://data.bs.ch/api/explore/v2.1/catalog/datasets/100089/records/?lang=en&limit=10&offset=0", headers=headers)
#Data received : Field
#  Identifier Type
#1	Zeitstempel	timestamp	Datetime	
#2	Abflussmenge	abfluss	Decimal	
#3	Pegel	pegelhoehe	Decimal	
#4	Wasserstand	pegel	Decimal	



In [3]:
import os
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests

DATASET_URL = "https://data.bs.ch/api/explore/v2.1/catalog/datasets/100089/records"
PAGE_SIZE = 100  # max allowed by the Explore API v2.1


def get_last_2_days(days=2, api_key=None):
    """Fetch the last `days` days of Rhein measurements from data.bs.ch (dataset 100089).

    Returns a DataFrame with columns: timestamp (UTC-aware), abfluss, pegelhoehe, pegel,
    sorted oldest -> newest. Records come at 5-minute intervals (~576 rows for 2 days).
    """
    api_key = api_key or os.environ.get("API_KEY")
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Apikey {api_key}"

    since = datetime.now(timezone.utc) - timedelta(days=days)

    rows, offset = [], 0
    while True:
        params = {
            "where": f"timestamp >= date'{since.strftime('%Y-%m-%dT%H:%M:%S')}'",
            "order_by": "timestamp asc",
            "limit": PAGE_SIZE,
            "offset": offset,
        }
        response = requests.get(DATASET_URL, headers=headers, params=params, timeout=30)
        response.raise_for_status()
        results = response.json().get("results", [])
        rows.extend(results)
        if len(results) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    df = pd.DataFrame(rows, columns=["timestamp", "abfluss", "pegelhoehe", "pegel"])
    if not df.empty:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
        df = df.sort_values("timestamp").reset_index(drop=True)
    return df


df = get_last_2_days()
print(f"{len(df)} rows from {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()
df.tail()

573 rows from 2026-10-01 09:10:00+00:00 to 2026-10-03 08:50:00+00:00


,timestamp,abfluss,pegelhoehe,pegel
568,2026-10-03 08:30:00+00:00,262.084,468.1,244.681
569,2026-10-03 08:35:00+00:00,284.769,471.0,244.710
570,2026-10-03 08:40:00+00:00,301.064,473.1,244.731
571,2026-10-03 08:45:00+00:00,324.926,476.2,244.762
572,2026-10-03 08:50:00+00:00,360.690,480.9,244.809


In [10]:
#Specific thresholds for Basel Rheinhalle station : 
#Critical, no navigation allowed, too high : pegelhoehe > 790 (Official Marke 2 restrictions)
#Warning, limited navigation (level 3), water level too high : pegelhoehe > 700 (Offical Marke 1 restrictions) 
#Warning, limited navigation (level 2), water level too high : pegelhoehe > 670 (boat length restricted - max 135 m)
#Warning, limited naviation (level 1), water level too high : pegelhoehe > 620 (boat length restricted - max 125 m)
#Warning, limited navigation, water level too low : pegelhoehe - 246 < 350 (max boat load not possible for ref boat with 350m draft)
#Critical, no navigation for economical reasons, water level too low : pegelhoehe - 246 <230 (less than 50% boat loat possible for ref boat with 350m draft)

In [13]:
from collections.abc import Mapping

import math

MARGIN_CM = 5.0  # fixed 5 cm band around every threshold
DRAFT_OFFSET = 246  # cm; available draft (Abladetiefe) = pegelhoehe - 246

# Basel Rheinhalle, high water. Checked against `pegelhoehe` [cm], most severe first.
HIGH_THRESHOLDS = [
    {"limit": 790, "severity": "critical", "label": "Marke 2",
     "consequence": "no navigation allowed"},
    {"limit": 700, "severity": "warning", "label": "level 3 (Marke 1)",
     "consequence": "limited navigation"},
    {"limit": 670, "severity": "warning", "label": "level 2",
     "consequence": "boat length limited to 135 m"},
    {"limit": 620, "severity": "warning", "label": "level 1",
     "consequence": "boat length limited to 125 m"},
]

# Basel Rheinhalle, low water. Checked against the available draft [cm], most severe first.
LOW_THRESHOLDS = [
    {"limit": 230, "severity": "critical", "label": "uneconomical",
     "consequence": "less than 50% load for the 350 cm draft reference boat"},
    {"limit": 350, "severity": "warning", "label": "reduced load",
     "consequence": "max load not possible for the 350 cm draft reference boat"},
]

_RANK = {"normal": 0, "watch": 1, "warning": 2, "critical": 3}


def _classify(value, limit, too_high):
    """Return (state, distance) where state is 'ok', 'margin' or 'breached'.

    'margin'   -> inside the 5 cm band but the threshold is not crossed yet
    'breached' -> past the threshold; `marginal` tells whether it is still inside the band
    """
    gap = (value - limit) if too_high else (limit - value)  # >0 means the limit is crossed
    if gap > 0:
        return "breached", gap
    if -gap <= MARGIN_CM:
        return "margin", abs(gap)
    return "ok", abs(gap)


def _check_family(value, thresholds, quantity, too_high):
    """Walk thresholds most-severe-first, stopping after the first crossed one."""
    findings = []
    for rule in thresholds:
        state, distance = _classify(value, rule["limit"], too_high)
        if state == "ok":
            continue
        marginal = state == "breached" and distance <= MARGIN_CM
        direction = "above" if too_high else "below"
        if state == "breached":
            text = (f"{quantity} {value:.1f} cm is {distance:.1f} cm {direction} the "
                    f"{rule['severity']} threshold of {rule['limit']} cm "
                    f"[{rule['label']}] -> {rule['consequence']}")
            if marginal:
                text += (f" (only just crossed, still inside the "
                         f"{MARGIN_CM:.0f} cm margin)")
        else:
            text = (f"{quantity} {value:.1f} cm is within the {MARGIN_CM:.0f} cm margin of the "
                    f"{rule['severity']} threshold of {rule['limit']} cm [{rule['label']}], "
                    f"{distance:.1f} cm to go -> would mean: {rule['consequence']}")
        findings.append({
            "quantity": quantity,
            "value": value,
            "limit": rule["limit"],
            "label": rule["label"],
            "severity": rule["severity"],
            "state": state,
            "marginal": marginal,
            "distance_cm": distance,
            "consequence": rule["consequence"],
            "message": text,
        })
        if state == "breached":
            break  # less severe thresholds below this one are implied
    return findings


def assess_conditions(measurement):
    """Judge one measurement against the Basel Rheinhalle navigation thresholds.

    `measurement` is a row of the DataFrame (dict or pandas Series), or a bare
    pegelhoehe value in cm.

    Returns a dict with:
      status   'normal' | 'watch' | 'warning' | 'critical'
      normal   True only when status == 'normal'
      findings list of everything that is not normal, most severe first
      summary  printable text
    """
    if isinstance(measurement, Mapping) or hasattr(measurement, "index"):
        pegelhoehe = measurement["pegelhoehe"]
        timestamp = measurement["timestamp"] if "timestamp" in measurement else None
    else:
        pegelhoehe, timestamp = measurement, None

    if pegelhoehe is None or (isinstance(pegelhoehe, float) and math.isnan(pegelhoehe)):
        return {"status": "unknown", "normal": False, "timestamp": timestamp,
                "pegelhoehe": None, "draft_cm": None, "findings": [],
                "summary": "No pegelhoehe in this measurement - cannot assess."}

    pegelhoehe = float(pegelhoehe)
    draft = pegelhoehe - DRAFT_OFFSET

    findings = (_check_family(pegelhoehe, HIGH_THRESHOLDS, "water level", too_high=True)
                + _check_family(draft, LOW_THRESHOLDS, "available draft", too_high=False))

    status = "normal"
    for f in findings:
        level = f["severity"] if f["state"] == "breached" else "watch"
        if _RANK[level] > _RANK[status]:
            status = level
    findings.sort(key=lambda f: -_RANK[f["severity"] if f["state"] == "breached" else "watch"])

    head = f"[{status.upper()}] water level {pegelhoehe:.1f} cm, available draft {draft:.1f} cm"
    if timestamp is not None:
        head += f" at {timestamp}"
    if status == "normal":
        body = ["Conditions are normal: no threshold crossed and none within the "
                f"{MARGIN_CM:.0f} cm margin."]
    else:
        body = [("  - CROSSED: " if f["state"] == "breached" else "  - NEAR:    ") + f["message"]
                for f in findings]

    return {
        "status": status,
        "normal": status == "normal",
        "timestamp": timestamp,
        "pegelhoehe": pegelhoehe,
        "draft_cm": draft,
        "findings": findings,
        "summary": "\n".join([head] + body),
    }


print(assess_conditions(df.iloc[-1])["summary"])


[WARNING] water level 480.9 cm, available draft 234.9 cm at 2026-10-03 08:50:00+00:00
  - CROSSED: available draft 234.9 cm is 115.1 cm below the warning threshold of 350 cm [reduced load] -> max load not possible for the 350 cm draft reference boat
  - NEAR:    available draft 234.9 cm is within the 5 cm margin of the critical threshold of 230 cm [uneconomical], 4.9 cm to go -> would mean: less than 50% load for the 350 cm draft reference boat


In [14]:
assess_conditions(df.iloc[-1])


{'status': 'warning',
 'normal': False,
 'timestamp': Timestamp('2026-10-03 08:50:00+0000', tz='UTC'),
 'pegelhoehe': 480.89999999999975,
 'draft_cm': 234.89999999999975,
 'findings': [{'quantity': 'available draft',
   'value': 234.89999999999975,
   'limit': 350,
   'label': 'reduced load',
   'severity': 'warning',
   'state': 'breached',
   'marginal': False,
   'distance_cm': 115.10000000000025,
   'consequence': 'max load not possible for the 350 cm draft reference boat',
   'message': 'available draft 234.9 cm is 115.1 cm below the warning threshold of 350 cm [reduced load] -> max load not possible for the 350 cm draft reference boat'},
  {'quantity': 'available draft',
   'value': 234.89999999999975,
   'limit': 230,
   'label': 'uneconomical',
   'severity': 'critical',
   'state': 'margin',
   'marginal': False,
   'distance_cm': 4.89999999999975,
   'consequence': 'less than 50% load for the 350 cm draft reference boat',
   'message': 'available draft 234.9 cm is within the 

In [ ]:
# Official BAFU ensemble forecast for Basel Rheinhalle (station 2289).
# The hydrodaten.admin.ch station page draws its forecast from a Plotly figure that is
# served as JSON - same numbers as the published plot, no scraping of pixels:
#   https://www.hydrodaten.admin.ch/plots/q_forecast/2289_q_forecast_en.json   (discharge, m3/s)
#   https://www.hydrodaten.admin.ch/plots/p_forecast/2289_p_forecast_en.json   (level, m a.s.l.)
# Traces: Median, Min/Max (two bounds), 25.-75. percentile (a closed polygon),
# and Measured for the preceding 24 h. Hourly steps, ~5 days ahead.

BAFU_STATION = 2289
BAFU_PLOT_URL = "https://www.hydrodaten.admin.ch/plots/{kind}/{station}_{kind}_{lang}.json"
BAFU_KINDS = {"discharge": "q_forecast", "level": "p_forecast"}
# Gauge zero at Basel Rheinhalle, verified against the data.bs.ch series:
# pegelhoehe[cm] == (pegel[m a.s.l.] - 240.00) * 100 exactly, for every row.
GAUGE_DATUM_M = 240.00


def _plotly_traces(fig):
    """Split the forecast figure into median / bounds / band / measured traces."""
    traces = fig["plot"]["data"]
    median = next(t for t in traces if "median" in (t.get("name") or "").lower())
    n = len(median["x"])
    measured, bounds, band = None, [], None
    for trace in traces:
        if trace is median:
            continue
        name = (trace.get("name") or "").lower()
        if any(k in name for k in ("measured", "gemessen", "mesur", "misur")):
            measured = trace
        elif len(trace["x"]) == 2 * n + 1:  # closed polygon: edge, reversed edge, close
            band = ([float(v) for v in trace["y"][:n]],
                    [float(v) for v in trace["y"][n:2 * n]][::-1])
        elif len(trace["x"]) == n:
            bounds.append([float(v) for v in trace["y"]])
    return median, bounds, band, measured


def get_bafu_forecast(station=BAFU_STATION, variable="level", lang="en",
                      datum_m=GAUGE_DATUM_M):
    """Fetch the official BAFU ensemble forecast for one station.

    variable 'level'     -> converted to pegelhoehe [cm], directly comparable to the
                            thresholds and to the data.bs.ch series
             'discharge' -> left in m3/s

    Returns a DataFrame (timestamp, median, p25, p75, min, max) with
    df.attrs holding 'title', 'issued', 'unit', 'station', 'variable' and
    'measured' (the observed last 24 h, same units).
    """
    if variable not in BAFU_KINDS:
        raise ValueError(f"variable must be one of {sorted(BAFU_KINDS)}")
    url = BAFU_PLOT_URL.format(kind=BAFU_KINDS[variable], station=station, lang=lang)
    response = requests.get(url, timeout=30)
    if response.status_code == 404:
        raise LookupError(f"no {variable} forecast published for station {station}")
    response.raise_for_status()
    fig = response.json()

    median, bounds, band, measured = _plotly_traces(fig)
    to_cm = (lambda v: (v - datum_m) * 100.0) if variable == "level" else (lambda v: v)

    out = pd.DataFrame({
        "timestamp": pd.to_datetime(median["x"], utc=True),
        "median": [to_cm(float(v)) for v in median["y"]],
    })
    if band:
        lo, hi = band
        out["p25"] = [to_cm(min(a, b)) for a, b in zip(lo, hi)]
        out["p75"] = [to_cm(max(a, b)) for a, b in zip(lo, hi)]
    if len(bounds) == 2:
        out["min"] = [to_cm(min(a, b)) for a, b in zip(*bounds)]
        out["max"] = [to_cm(max(a, b)) for a, b in zip(*bounds)]

    observed = pd.DataFrame(columns=["timestamp", "value"])
    if measured is not None:
        observed = pd.DataFrame({
            "timestamp": pd.to_datetime(measured["x"], utc=True),
            "value": [to_cm(float(v)) for v in measured["y"]],
        }).dropna()

    layout = fig["plot"]["layout"]
    issued = next((a.get("text") for a in layout.get("annotations", [])
                   if "forecast" in (a.get("text") or "").lower()
                   or "vorhersage" in (a.get("text") or "").lower()), None)
    out.attrs.update({
        "title": layout.get("title"),
        "issued": issued,
        "unit": "cm (pegelhoehe)" if variable == "level" else "m3/s",
        "station": station,
        "variable": variable,
        "source": url,
        "measured": observed,
    })
    return out


def forecast_conditions_bafu(df=None, days_ahead=5, station=BAFU_STATION, lang="en"):
    """Assess now vs. `days_ahead` ahead using the official BAFU ensemble forecast.

    Returns `latest` and `forecast` (each with timestamp, pegelhoehe and its
    assessment), `band` (the assessment of the ensemble envelope at the horizon),
    `first_crossings` (when the median first reaches each status), the full
    `series`, and a printable `summary`.
    """
    forecast = get_bafu_forecast(station=station, variable="level", lang=lang)
    if forecast.empty:
        raise LookupError("forecast came back empty")

    if df is not None and not df.empty:
        latest_row = df.dropna(subset=["pegelhoehe"]).sort_values("timestamp").iloc[-1]
        latest = {"timestamp": latest_row["timestamp"],
                  "pegelhoehe": float(latest_row["pegelhoehe"])}
        source_now = "data.bs.ch"
    else:
        observed = forecast.attrs["measured"]
        latest = {"timestamp": observed["timestamp"].iloc[-1],
                  "pegelhoehe": float(observed["value"].iloc[-1])}
        source_now = "BAFU measured"

    wanted = latest["timestamp"] + pd.Timedelta(days=days_ahead)
    horizon_end = forecast["timestamp"].iloc[-1]
    truncated = wanted > horizon_end
    row = forecast.iloc[(forecast["timestamp"] - wanted).abs().argmin()]

    latest_assessment = assess_conditions(latest)
    forecast_assessment = assess_conditions({"timestamp": row["timestamp"],
                                             "pegelhoehe": float(row["median"])})
    band = {}
    for key in ("min", "p25", "p75", "max"):
        if key in forecast.columns:
            band[key] = assess_conditions({"timestamp": row["timestamp"],
                                           "pegelhoehe": float(row[key])})

    statuses = [assess_conditions(float(v))["status"] for v in forecast["median"]]
    first_crossings = {}
    for status, timestamp in zip(statuses, forecast["timestamp"]):
        first_crossings.setdefault(status, timestamp)

    lines = [
        f"BAFU station {station} - {forecast.attrs['title']}",
        f"{forecast.attrs['issued']}, horizon {forecast['timestamp'].iloc[0]} -> {horizon_end}",
        f"Now ({source_now}) {latest['pegelhoehe']:.1f} cm -> "
        f"{row['median']:.1f} cm median at {row['timestamp']}",
    ]
    if band:
        lines.append("  ensemble at that time: "
                     + ", ".join(f"{k} {float(row[k]):.1f} cm [{band[k]['status']}]"
                                 for k in band))
    if truncated:
        lines.append(f"NOTE: the forecast only reaches {horizon_end}, short of the "
                     f"{days_ahead}-day horizon - reporting its last step instead.")
    if first_crossings:
        lines.append("  median status timeline, first occurrence of each "
                     "(the earliest may already hold at the forecast start): "
                     + ", ".join(f"{s} at {t}" for s, t in first_crossings.items()))
    summary = "\n".join(lines + [
        "", "NOW:", latest_assessment["summary"],
        "", f"AT {row['timestamp']}:", forecast_assessment["summary"],
    ])

    return {
        "latest": {**latest, "assessment": latest_assessment},
        "forecast": {"timestamp": row["timestamp"],
                     "pegelhoehe": float(row["median"]),
                     "assessment": forecast_assessment},
        "band": band,
        "first_crossings": first_crossings,
        "series": forecast,
        "summary": summary,
    }


print(forecast_conditions_bafu(df)["summary"])
